# TMDB movie analysis

I fetch 18 movies, clean them, check profit and ratings, then draw charts.

In [ ]:
import pandas as pd
from tmdb_client import fetch_all

records = fetch_all()
df = pd.DataFrame(records)
print(len(df))
df[["title", "budget", "revenue"]].head()

## clean

Drop bad columns, join lists with |, fix types, dollars to millions, keep Released.

In [ ]:
from src.cleaning import clean

df2 = clean(df)
print(df2.shape)
df2[["title", "genres", "budget_musd", "revenue_musd"]].head()

## check the extracted columns

The brief asks for value_counts on the columns I built, to spot odd values before I trust them.

In [ ]:
from src.cleaning import inspect_anomalies

EXTRACTED = ["belongs_to_collection", "genres", "spoken_languages",
              "production_countries", "production_companies", "original_language"]
counts = inspect_anomalies(df2, EXTRACTED)

## profit and roi

In [ ]:
from src.kpis import add_profit_roi, rank

df3 = add_profit_roi(df2)
print(df3[["title", "profit", "roi"]].head())

print("top revenue")
print(rank(df3, "revenue_musd")["title"].tolist())
print("top profit")
print(rank(df3, "profit")["title"].tolist())
print("low profit")
print(rank(df3, "profit", ascending=True)["title"].tolist())

## more rankings

In [ ]:
print("top budget")
print(rank(df3, "budget_musd")["title"].tolist())
print("most voted")
print(rank(df3, "vote_count")["title"].tolist())
print("most popular")
print(rank(df3, "popularity")["title"].tolist())

rich = df3[df3["budget_musd"] >= 10]
print("top roi")
print(rank(rich, "roi")["title"].tolist())
print("low roi")
print(rank(rich, "roi", ascending=True)["title"].tolist())

voted = df3[df3["vote_count"] >= 10]
print("top rated")
print(rank(voted, "vote_average")["title"].tolist())
print("low rated")
print(rank(voted, "vote_average", ascending=True)["title"].tolist())

## searches

In [ ]:
from src.kpis import search_willis_scifi_action, search_thurman_tarantino

w = search_willis_scifi_action(df3)
print("willis found:", len(w))
print(w[["title", "vote_average"]] if len(w) > 0 else "no match in this set")

t = search_thurman_tarantino(df3)
print("thurman found:", len(t))
print(t[["title", "runtime"]] if len(t) > 0 else "no match in this set")

## franchise and directors

In [ ]:
from src.kpis import franchise_vs_standalone, top_franchises, top_directors

print(franchise_vs_standalone(df3))
print(top_franchises(df3).head())
print(top_directors(df3).head())

## charts

In [ ]:
import matplotlib.pyplot as plt
from src.visualize import plot_revenue_vs_budget, plot_roi_by_genre, plot_popularity_vs_rating, plot_yearly_box_office, plot_franchise_vs_standalone

plot_revenue_vs_budget(df3)
plt.show()

plot_roi_by_genre(df3)
plt.tight_layout()
plt.show()

plot_popularity_vs_rating(df3)
plt.show()

plot_yearly_box_office(df3)
plt.show()

plot_franchise_vs_standalone(df3)
plt.show()